## Xử lý, tính return BTC-VND

In [ ]:
import pandas as pd
import numpy as np

btc = pd.read_csv(
    "3y-bitcoin-price-clean.csv",
    parse_dates=["Date"]
)

fx = pd.read_csv(
    "3y-usdvnd-price-clean.csv",
    parse_dates=["Date"]
)

# Sorted by Date
btc = btc.sort_values("Date").reset_index(drop=True)
fx = fx.sort_values("Date").reset_index(drop=True)

print("BTC:")
print("Rows:", len(btc))
print("Date range:", btc["Date"].min(), "->", btc["Date"].max())
print("Duplicated dates:", btc["Date"].duplicated().sum())

print("\nUSD/VND:")
print("Rows:", len(fx))
print("Date range:", fx["Date"].min(), "->", fx["Date"].max())
print("Duplicated dates:", fx["Date"].duplicated().sum())

btc_vnd = btc.merge(
    fx[["Date", "usdvnd_rate"]],
    on="Date",
    how="left",
    validate="one_to_one"
)


# MARK WHETHER THE FX RATE IS OBSERVED OR FILLED

btc_vnd["fx_observed"] = btc_vnd["usdvnd_rate"].notna()


# FORWARD-FILL MISSING USD/VND RATES
# For weekends / holidays, use the latest available USD/VND rate.
#
# Example:
# Friday     26,200
# Saturday      NaN -> 26,200
# Sunday        NaN -> 26,200
# Monday     26,230

btc_vnd["usdvnd_rate_used"] = (
    btc_vnd["usdvnd_rate"]
    .ffill()
)
# CHECK THAT NO BTC OBSERVATIONS WERE LOST

assert len(btc_vnd) == len(btc), \
    "Error: BTC observations were lost during merge."

assert btc_vnd["Date"].equals(btc["Date"]), \
    "Error: BTC calendar changed during merge."

assert btc_vnd["usdvnd_rate_used"].isna().sum() == 0, \
    "Error: Some dates still have no usable USD/VND rate."


# CONVERT BTC-USD PRICE TO BTC-VND PRICE

btc_vnd["btc_vnd_price"] = (
    btc_vnd["btc_price"]
    * btc_vnd["usdvnd_rate_used"]
)


# CALCULATE LOG RETURN
# Multiply by 100 so returns are expressed in percentage terms.

btc_vnd["btc_return"] = (
    100
    * np.log(
        btc_vnd["btc_vnd_price"]
        / btc_vnd["btc_vnd_price"].shift(1)
    )
)

# 10. OPTIONAL: BTC-USD RETURN FOR COMPARISON

btc_vnd["btc_usd_return"] = (
    100
    * np.log(
        btc_vnd["btc_price"]
        / btc_vnd["btc_price"].shift(1)
    )
)


# IDENTIFY DAYS WHERE FX WAS FORWARD-FILLED

btc_vnd["fx_filled"] = ~btc_vnd["fx_observed"]


# FINAL CHECKS

print("\nFinal dataset")
print("Rows:", len(btc_vnd))
print(
    "Date range:",
    btc_vnd["Date"].min(),
    "->",
    btc_vnd["Date"].max()
)

print(
    "Missing BTC-VND prices:",
    btc_vnd["btc_vnd_price"].isna().sum()
)

print(
    "Missing returns:",
    btc_vnd["btc_return"].isna().sum()
)

print(
    "Number of FX rates forward-filled:",
    btc_vnd["fx_filled"].sum()
)

print("\nSample of filled FX dates:")
print(
    btc_vnd.loc[
        btc_vnd["fx_filled"],
        [
            "Date",
            "btc_price",
            "usdvnd_rate",
            "usdvnd_rate_used",
            "btc_vnd_price",
            "btc_return"
        ]
    ].head(15)
)


# SAVE

btc_vnd.to_csv(
    "3y-bitcoin-vnd-returns.csv",
    index=False
)

BTC:
Rows: 1097
Date range: 2023-08-01 00:00:00 -> 2026-08-01 00:00:00
Duplicated dates: 0

USD/VND:
Rows: 779
Date range: 2023-08-01 00:00:00 -> 2026-07-31 00:00:00
Duplicated dates: 0

Final dataset
Rows: 1097
Date range: 2023-08-01 00:00:00 -> 2026-08-01 00:00:00
Missing BTC-VND prices: 0
Missing returns: 1
Number of FX rates forward-filled: 318

Sample of filled FX dates:
         Date     btc_price  usdvnd_rate  usdvnd_rate_used  btc_vnd_price  \
4  2023-08-05  29042.126953          NaN           23750.0   6.897505e+08   
5  2023-08-06  29041.855469          NaN           23750.0   6.897441e+08   
11 2023-08-12  29415.964844          NaN           23740.0   6.983350e+08   
12 2023-08-13  29282.914062          NaN           23740.0   6.951764e+08   
18 2023-08-19  26096.205078          NaN           23850.0   6.223945e+08   
19 2023-08-20  26189.583984          NaN           23850.0   6.246216e+08   
25 2023-08-26  26008.462891          NaN           23900.0   6.216023e+08   
26 20

In [ ]:
filled_days = btc_vnd["fx_filled"]

check = (
    btc_vnd.loc[filled_days, "btc_return"]
    - btc_vnd.loc[filled_days, "btc_usd_return"]
)

print(check.abs().max())

0.0


## Tính return giá vàng SJC

In [ ]:
import pandas as pd
import numpy as np

gold = pd.read_csv(
    "3y-sjc-price-clean.csv",
    parse_dates=["Date"]
)

gold = gold.sort_values("Date").reset_index(drop=True)

gold["gold_return"] = (
    100 * np.log(
        gold["gold_price"]
        / gold["gold_price"].shift(1)
    )
)
gold.to_csv(
    "3y-gold-returns.csv",
    index=False
)

## Tính return VN-Index

In [ ]:
import pandas as pd
import numpy as np

vnindex = pd.read_csv(
    "3y-vnindex-data-clean.csv",
    parse_dates=["Date"]
)

vnindex = vnindex.sort_values("Date").reset_index(drop=True)

vnindex["vnindex_return"] = (
    100 * np.log(
        vnindex["vnindex_price"]
        / vnindex["vnindex_price"].shift(1)
    )
)
vnindex.to_csv(
    "3y-vnindex-returns.csv",
    index=False
)